# HuggingFace dataset types and formats

[Reference](https://huggingface.co/docs/trl/en/dataset_formats#dataset-formats-and-types)


In the `trl` library, the **Type** determines the objective of the training, while the **Format** describes the structure

- Standard format is flat text
- Conversational format is structured text


# Dataset Types (Standard Format)

## Language Modeling Type

* **Goal:** Next-token prediction on the entire sequence.
* **Required Column:** `text` (or any single string column).
* **Use Case:** Pre-training on raw data or basic SFT where you don't mind the model "learning" the user's prompt tokens.

In [6]:
# Standard Format: Language Modeling Type
lm_data = [
    {"text": "The capital of France is Paris and it is known for the Eiffel Tower."},
    {"text": "Quantum mechanics is a fundamental theory in physics that describes nature."}
]

## Prompt-Completion Type

* **Goal:** Generating a specific answer based on a specific input.
* **Required Columns:** `prompt` and `completion`.
* **Use Case:** The most common format for Instruction Tuning. This allows for **Strategy C: Architect Masking**, where the loss is only calculated on the `completion`.

In [7]:
# Standard Format: Prompt-Completion Type
sft_data = [
    {"prompt": "What is the square root of 144?", "completion": "The square root of 144 is 12."},
    {"prompt": "Summarize: The sun is a star at the center of the Solar System.", "completion": "The sun is the star at our system's center."}
]

## Preference Type

* **Goal:** Learning to rank responses (comparative learning).
* **Required Columns:** `prompt`, `chosen`, and `rejected`.
* **Use Case:** DPO (Direct Preference Optimization). It teaches the model to favor the "chosen" string over the "rejected" string given the same "prompt."

In [8]:
# Standard Format: Preference Type
dpo_data = [
    {
        "prompt": "Write a polite greeting.",
        "chosen": "Hello! It is a pleasure to meet you today.",
        "rejected": "Hey. What do you want?"
    },
    {
        "prompt": "How do I reset my password?",
        "chosen": "You can reset your password by clicking 'Forgot Password' on the login page.",
        "rejected": "I don't know, look it up on Google."
    }
]

## **Quick Jupyter Setup**

You can paste this into a cell to create these as Hugging Face `Dataset` objects immediately:

In [9]:
from datasets import Dataset

# Create the datasets
ds_lm = Dataset.from_list(lm_data)
ds_sft = Dataset.from_list(sft_data)
ds_dpo = Dataset.from_list(dpo_data)

# Print one example to verify
print("SFT Example:", ds_sft[0])

SFT Example: {'prompt': 'What is the square root of 144?', 'completion': 'The square root of 144 is 12.'}



## **Which one should you use?**

* **Use Type 1 (LM)** if you have a massive pile of raw text and just want the model to learn the "vibe" or style of that text.
* **Use Type 2 (Prompt-Completion)** for your SFT and GRPO reasoning tasks. It provides the cleanest separation of concerns for your reward functions and loss masking.
* **Use Type 3 (Preference)** only if you are doing alignment training to fix specific model behaviors (like stopping the model from being rude or repetitive).

# Dataset types (Conversational format)

To move from the **Standard Format** to the **Conversational Format**, we shift from flat strings to structured lists of messages.

In the Hugging Face `trl` ecosystem, this structure is highly preferred because it allows the model to differentiate between the **System** (instructions), the **User** (queries), and the **Assistant** (target answers).

Below are the three major **Types** organized in the **Conversational Format**, ready for use in a Jupyter Notebook.




## Language Modeling Type

* **Required Column:** `messages`.
* **Structure:** A single list containing the entire dialogue history.
* **Training Logic:** The model learns from the entire sequence.

In [10]:
# Conversational Format: Language Modeling Type
lm_conv_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful travel guide."},
            {"role": "user", "content": "What is the best time to visit Tokyo?"},
            {"role": "assistant", "content": "Spring (March to May) is ideal for cherry blossoms."}
        ]
    }
]

## Prompt-Completion Type

* **Required Columns:** `prompt` and `completion`.
* **Structure:** Both keys contain a list of messages. The `completion` is typically the final assistant turn.
* **Training Logic:** Perfectly supports **Architect Masking**. The trainer applies the chat template to both, but only calculates loss on the `completion` tokens.

In [11]:
# Conversational Format: Prompt-Completion Type
sft_conv_data = [
    {
        "prompt": [
            {"role": "system", "content": "You are a math tutor."},
            {"role": "user", "content": "Solve for x: 2x = 10"}
        ],
        "completion": [
            {"role": "assistant", "content": "Dividing both sides by 2, we find x = 5."}
        ]
    }
]

## Preference Type

* **Required Columns:** `prompt`, `chosen`, and `rejected`.
* **Structure:** All three keys use the message list format. The `prompt` provides the context, while `chosen` and `rejected` provide the competing assistant responses.

In [12]:
# Conversational Format: Preference Type
dpo_conv_data = [
    {
        "prompt": [
            {"role": "system", "content": "You are a polite assistant."},
            {"role": "user", "content": "Tell me a joke."}
        ],
        "chosen": [
            {"role": "assistant", "content": "Why did the scarecrow win an award? Because he was outstanding in his field!"}
        ],
        "rejected": [
            {"role": "assistant", "content": "I don't have time for jokes right now."}
        ]
    }
]


## Quick Jupyter Setup

You can use the code below to convert these lists into Hugging Face `Dataset` objects:

In [13]:
from datasets import Dataset

# Create the conversational datasets
ds_lm_conv = Dataset.from_list(lm_conv_data)
ds_sft_conv = Dataset.from_list(sft_conv_data)
ds_dpo_conv = Dataset.from_list(dpo_conv_data)

# Print a sample to see the nested structure
import pprint
pprint.pprint(ds_sft_conv[0])

{'completion': [{'content': 'Dividing both sides by 2, we find x = 5.',
                 'role': 'assistant'}],
 'prompt': [{'content': 'You are a math tutor.', 'role': 'system'},
            {'content': 'Solve for x: 2x = 10', 'role': 'user'}]}



## **Which one should you use?**

* **The Power of System Prompts:** Use the Conversational Format specifically when you want to use a `system` role to "anchor" the model's behavior (e.g., "You are a concise scientist").
* **The GRPO Advantage:** When moving to Reinforcement Learning, the **Prompt-Completion Type** in the **Conversational Format** is the gold standard.

It allows the model to understand the conversation flow (`prompt`) while giving your reward functions a structured reference (`completion`) to check against.



## Chat templates: converting from Conversational format to "raw" text


When you use the **Standard Format** (flat strings), the model sees exactly what you typed.

When you use the **Conversational Format**, the data is actually **incomplete** until it passes through a **Chat Template**.



Here is the breakdown of why this happens and why it’s actually a "feature," not a bug.


## **The "Hidden" Translation**

The LLM does not actually know what a "dictionary" or a "role" is.

It only understands a long string of text.

The **Chat Template** (usually written in Jinja2) acts as the translator that turns your structured list into the specific "language" the model was trained on.

### **The Visual Transformation**

If you are using **Llama 3.1**, your conversational data looks like this in your code:
`{"role": "user", "content": "Hi"}`

But after the **Chat Template** processes it, the model actually sees:
`<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHi<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n`

---

## **Why this is Mandatory for Modern Models**

If you skip the chat template and just feed the model raw strings (Standard Format), you lose the **Special Control Tokens** (like `<|eot_id|>`).

* **Without Templates:** The model might get confused about when the User stopped talking and when it should start.
* **With Templates:** The model recognizes the "Header IDs" and knows exactly how to structure its internal attention.





### How to handle this in Jupyter

The good news is that Hugging Face makes this nearly automatic. You don't have to write the Jinja2 code yourself; you just call the model's tokenizer.

### **The "Manual" Way (To see what's happening)**

In [15]:
from transformers import AutoTokenizer

# Qwen 2.5 is open, small (0.5B), and perfect for testing in Colab
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Our Conversational Format data
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain 2+2."}
]

# 1. Transform list of dictionaries into a raw string
# tokenize=False lets us see the human-readable string
# add_generation_prompt=True adds the 'assistant' header to trigger a response
templated_string = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("--- RAW TEMPLATED STRING ---")
print(templated_string)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- RAW TEMPLATED STRING ---
<|im_start|>system
You are a concise assistant.<|im_end|>
<|im_start|>user
Explain 2+2.<|im_end|>
<|im_start|>assistant

